# Extract Routing Trace from TinyMoE-100m-2x8

**Corrected version** (fixes experts[0, pos] → experts[pos])

This notebook:
1. Installs Rust 1.93.0
2. Runs corrected extract_routing_trace.py
3. Downloads the trace
4. Optionally runs forge bench-read

In [ ]:
# Install Rust 1.93.0
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.93.0
!rustup default 1.93.0
!rustc --version

In [ ]:
# Install dependencies for trace extraction
!pip install torch transformers accelerate

In [ ]:
# ============================================================
# extract_routing_trace.py — Colab-ready (CORRECTED)
# ============================================================
# Извлекает routing trace из FlameF0X/TinyMoE-100m-2x8.
#
# Выход: /content/routing-trace.jsonl
# Формат: одна строка на (layer, token_pos):
#   {"layer": 0, "pos": 0, "experts": [3, 7], "weights": [0.62, 0.38]}
#
# Почему experts[pos], а не experts[0, pos]:
#   gate возвращает top_k_index формы [batch*seq, top_k].
#   experts[pos] — это вектор из top_k экспертов для токена pos.
#   experts[0, pos] — это скаляр (int), sorted() на нём падает.
# ============================================================

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

MODEL_ID = "FlameF0X/TinyMoE-100m-2x8"
NUM_TOKENS = 2000
OUT_PATH = "/content/routing-trace.jsonl"

# ------------------------------------------------------------
# 1. Загрузка модели и токенизатора
# ------------------------------------------------------------
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   # явно, без deprecation warning
    device_map="auto",           # CPU на Colab, если GPU нет
)
model.eval()

config = model.config
print(f"num_hidden_layers = {config.num_hidden_layers}")
print(f"num_experts = {getattr(config, 'num_local_experts', getattr(config, 'num_experts', '?'))}")
print(f"num_experts_per_tok = {getattr(config, 'num_experts_per_tok', '?')}")

# ------------------------------------------------------------
# 2. Хранилище трассировки
# ------------------------------------------------------------
trace = []                    # список dict, потом пишем в JSONL
_current_layer = {"idx": None}  # заполняется при регистрации хука

# ------------------------------------------------------------
# 3. Hook — исправленная версия
# ------------------------------------------------------------
def make_hook(layer_idx):
    def hook(module, args, output):
        # gate возвращает (router_logits, top_k_weights, top_k_index)
        # но в некоторых версиях transformers — просто router_logits.
        # Обрабатываем оба случая.
        if isinstance(output, tuple) and len(output) >= 3:
            _, top_k_weights, top_k_index = output[0], output[1], output[2]
        elif isinstance(output, tuple) and len(output) == 2:
            # Некоторые версии: (router_logits, top_k_index)
            top_k_index = output[1]
            top_k_weights = None
        else:
            # Только logits — вычисляем top-k сами
            router_logits = output if not isinstance(output, tuple) else output[0]
            top_k = getattr(module, "top_k", 2)
            top_k_weights, top_k_index = torch.topk(
                torch.softmax(router_logits, dim=-1), top_k, dim=-1
            )

        # ---- Ключевое исправление ----
        # top_k_index: [batch*seq, top_k]  (или [batch, seq, top_k])
        if top_k_index.dim() == 3:
            top_k_index = top_k_index.reshape(-1, top_k_index.shape[-1])
        if top_k_weights is not None and top_k_weights.dim() == 3:
            top_k_weights = top_k_weights.reshape(-1, top_k_weights.shape[-1])

        n_tokens = top_k_index.shape[0]

        # Диагностика: печатаем форму только для первого слоя и первого вызова
        if layer_idx == 0 and _current_layer["idx"] is None:
            print(f"[hook] layer={layer_idx} "
                  f"index_shape={tuple(top_k_index.shape)} "
                  f"weights_shape={tuple(top_k_weights.shape) if top_k_weights is not None else None}")

        for pos in range(n_tokens):
            exp_list = sorted(top_k_index[pos].tolist())
            w_list = (
                top_k_weights[pos].tolist()
                if top_k_weights is not None
                else [None] * len(exp_list)
            )
            trace.append({
                "layer": layer_idx,
                "pos": pos,
                "experts": exp_list,
                "weights": w_list,
            })

        _current_layer["idx"] = layer_idx

    return hook

# ------------------------------------------------------------
# 4. Регистрация хуков на всех gate-слоях
# ------------------------------------------------------------
hooks = []
for name, module in model.named_modules():
    if name.endswith(".mlp.gate"):
        # Извлекаем индекс слоя из имени: model.layers.N.mlp.gate
        parts = name.split(".")
        layer_idx = None
        for i, p in enumerate(parts):
            if p == "layers" and i + 1 < len(parts):
                try:
                    layer_idx = int(parts[i + 1])
                except ValueError:
                    pass
                break
        if layer_idx is None:
            continue
        h = module.register_forward_hook(make_hook(layer_idx))
        hooks.append(h)
        print(f"Hooked layer {layer_idx}: {name}")

print(f"Total hooks: {len(hooks)}")

# ------------------------------------------------------------
# 5. Генерация токенов и forward pass
# ------------------------------------------------------------
prompt = "The quick brown fox jumps over the lazy dog. " * 32
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"Generating {NUM_TOKENS} tokens...")
with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=NUM_TOKENS,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(f"Generated sequence length: {generated.shape[1]}")

# Теперь forward pass по всей последовательности, чтобы hooks сработали
# на всех токенах. generate() уже вызвал forward, но мы хотим явный контроль.
print("Running forward pass over the full sequence...")
with torch.no_grad():
    _ = model(generated)

# ------------------------------------------------------------
# 6. Удаление хуков
# ------------------------------------------------------------
for h in hooks:
    h.remove()
print("Hooks removed.")

# ------------------------------------------------------------
# 7. Запись JSONL
# ------------------------------------------------------------
# Группируем по (layer, pos), но слои уже идут в порядке срабатывания.
# Порядок в trace: layer 0 все токены, layer 1 все токены, ...
trace.sort(key=lambda r: (r["layer"], r["pos"]))

with open(OUT_PATH, "w") as f:
    for rec in trace:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(trace)} records to {OUT_PATH}")

# ------------------------------------------------------------
# 8. Быстрая сводка — распределение экспертов
# ------------------------------------------------------------
from collections import Counter

per_layer = {}
for rec in trace:
    per_layer.setdefault(rec["layer"], Counter()).update(rec["experts"])

print("\nExpert frequency per layer (top 3):")
for layer in sorted(per_layer):
    top = per_layer[layer].most_common(3)
    print(f"  layer {layer}: {top}")

# ------------------------------------------------------------
# 9. Скачивание файла
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(OUT_PATH)
except ImportError:
    print(f"(not in Colab; file at {OUT_PATH})")

In [ ]:
# Optional: build forge and run bench-read
!cargo build --release -p forge-cli
# Upload manifest.json from your machine, then:
# !./target/release/forge bench-read --manifest manifest.json --trace routing-trace.jsonl --direct --out bench-real.jsonl